# 03 - Train a DQN Model Online

Collect `EPISODES_PER_STREAM` complete outer tasks from each FrozenLake stream, then run
`TRAIN_STEPS` updates. Environment and agent events accumulate in per-stream `Datastore`
replay buffers; the loader samples whole tasks for the DQN objective.

Online and offline training use the same source events, model interface, and data
pipeline. This is a short usage example, not a full experiment. Evaluate the saved
checkpoint in `15_inference.ipynb`.


In [ ]:
from copy import deepcopy

import gymnasium as gym
import torch
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    DataLoader,
    Datastore,
    EventLog,
    Tokenizer,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    action_readouts,
    prepare_decode,
    push_model_to_hub,
    select_discrete_action,
)
from mouse_core.objectives import DqnObjective, boundary_discount

# Repositories
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-online"

# Environment
# number of discrete actions predicted by the head
MAX_ACTIONS = 4
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20
# number of environment streams in the GroupEnv
NUM_ENVS = 4

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final prediction linear's initialization
HEAD_SCALE = 0.1

# Collection and rollout
# complete outer tasks collected per stream before training
EPISODES_PER_STREAM = 1
MAX_CONTEXT = REPEAT_COUNT * (
    MAX_STEPS_PER_EPISODE + 1
)  # every inner step and reset in one outer task
# environment outputs collected before the first optimizer update
LEARNING_STARTS = 64
# env-step horizon for epsilon decay
EXPLORATION_ENDS = 1000

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer rollout/train cycles
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 20

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# delayed value head
POLYAK_TAU_VALUE = 0.001

# Runtime
ensure_registered()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Build Environment

Procedural FrozenLake is wrapped in `RepeatTaskEnv`, then in mouse-gym. `_terminated`
and `_truncated` identify outer task endings; the payload also records inner episode
flags. Each stream’s standalone `RepeatTaskTracker` consumes completed outputs to track
outer tasks, inner episodes, and completed statistics.

Training uses complete tasks and zero discount at their actual endings. The objective
raises if a sampled target needs a missing continuation.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode=None,
            emit_q_star=False,
        )
        env = RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
        return env

    return env_fn


configs = [
    EnvConfig(
        name=f"proc_frozenlake_online_{i}",
        seed=i,
        seed_mode="stream",
        env_fn=make_lake(stream=i),
        reset_options={"regenerate_map": True},
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

logs = [EventLog(name=name) for name in env.names]
trackers = [RepeatTaskTracker() for _ in logs]

## Data Pipeline

Each stream appends events to its own replay `Datastore`. Its policy context contains
the full current outer task, including inner resets. Collection finishes all requested
tasks before training, so replay contains no unfinished task tails.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


stores = [Datastore(name=name) for name in env.names]
contexts = [[] for _ in env.names]

# Online decode / replay: tokenizer (no augmenter)
episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)


transform = tokenizer.encode

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

The delayed encoder and Q head(s) are frozen copies updated with Polyak averaging.
The same delayed Q-values select and evaluate backup actions (plain DQN). Only
online parameters enter the optimizer.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_mask=tensors["head_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)
q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Collect Complete Tasks

Collect one stream at a time, finishing each outer task before moving to the next.
Choose a random or greedy action only when the environment state requests one. Greedy
inference uses a KV cache and appends observations not yet encoded, including
observations collected during random exploration.

Each task starts a fresh context and cache. `MAX_CONTEXT` must fit the worst-case task,
counting every inner reset; the rollout never truncates its history.


In [ ]:
def epsilon_for_env_step(*, env_step: int) -> float:
    """Return the epsilon-greedy exploration rate for a global env step."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    epsilon = 1.0 - frac
    return epsilon

In [ ]:
@torch.no_grad()
def run_rollout(*, env_steps: int, num_episodes: int, max_cache: int) -> int:
    """Collect complete outer tasks, one stream at a time, before training."""
    if num_episodes < 1:
        raise ValueError("num_episodes must be positive")
    if max_cache < REPEAT_COUNT * (MAX_STEPS_PER_EPISODE + 1):
        raise ValueError(
            "max_cache must fit a complete outer task, including reset observations"
        )
    modules.eval()
    for i, stream in enumerate(env.envs):
        log = logs[i]
        tracker = trackers[i]
        for _episode in range(num_episodes):
            contexts[i].clear()
            cache = None
            cached_rows = 0
            try:
                while True:
                    event_start = len(log)
                    episode_start = tracker.output is None or tracker.ended
                    episode_index = tracker.next_episode_index
                    if episode_start:
                        coordinator_event = log.event(
                            provider="coordinator",
                            episode_index=episode_index,
                            step_index=(0 if episode_start else tracker.step_index + 1),
                            payload={
                                "_episode_start": True,
                            },
                        )
                        stores[i].append(data=coordinator_event)
                    action = None
                    if tracker.action_requested:
                        epsilon = epsilon_for_env_step(env_step=env_steps)
                        if torch.rand(1).item() < epsilon:
                            action = int(stream.action_space.sample())
                        else:
                            pending = [contexts[i][cached_rows:]]
                            inputs, inputs_objective_data = tokenizer.pack_rows(
                                rows=pending,
                            )
                            out = encoder(inputs, cache=cache, use_cache=True)
                            cache = out["cache"]
                            cached_rows = len(contexts[i])
                            action = int(
                                select_discrete_action(
                                    scores=action_readouts(q_head(out["z"]), inputs),
                                    temperature=0.0,
                                    num_actions=MAX_ACTIONS,
                                )[0].item()
                            )
                        agent_event = log.event(
                            provider="agent",
                            episode_index=episode_index,
                            step_index=tracker.step_index + 1,
                            payload={
                                "action": action,
                            },
                        )
                        stores[i].append(data=agent_event)
                    output = stream.step(action)
                    tracker.update(output=output)

                    environment_event = log.event(
                        provider="environment",
                        episode_index=tracker.episode_index,
                        step_index=tracker.step_index,
                        payload={
                            "observation": tracker.observation,
                            **(
                                {"repeat_episode_index": tracker.repeat_episode_index}
                                if tracker.repeat_episode_start
                                else {}
                            ),
                            **(
                                {"inner_terminated": True}
                                if tracker.inner_terminated
                                else {}
                            ),
                            **(
                                {"inner_truncated": True}
                                if tracker.inner_truncated
                                else {}
                            ),
                            **(
                                {"_q_star": tracker.info["q_star"]}
                                if tracker.action_requested and "q_star" in tracker.info
                                else {}
                            ),
                            **({"_terminated": True} if tracker.terminated else {}),
                            **({"_truncated": True} if tracker.truncated else {}),
                            **(
                                {"reward": tracker.reward}
                                if tracker.reward != 0
                                else {}
                            ),
                            **(
                                {"action_requested": True}
                                if tracker.action_requested
                                else {}
                            ),
                        },
                    )
                    stores[i].append(data=environment_event)
                    contexts[i].extend(log[event_start:])
                    env_steps += 1
                    if tracker.step_index + 1 > max_cache:
                        raise ValueError(
                            "Environment exceeded the configured complete-task context bound"
                        )
                    if tracker.ended:
                        break
            finally:
                if cache is not None:
                    cache.close()
    return env_steps

## Train

The loader samples complete `episode_index` runs and fills `TOKEN_BUDGET` with whole
tasks. A task that cannot fit an empty batch raises. `weight_mode="per_event"` weights
streams by their source event counts. Every sampled task ends at its recorded
termination or truncation; the objective rejects any missing continuation.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_mask`) and `objective_data` (training labels and step mappings).
`objective_data["step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    weight_mode="per_event",
    num_workers=4,
    transform=tokenizer.encode,
)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    gate=None,
)
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Refresh replay and run ``num_steps`` optimizer updates."""
    modules.train()
    loader.refresh()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            backup_value = backup_value_head(delayed_out["z"])
        losses, metrics = objective(
            objective_data=objective_data.to(device),
            sequence_ids=sequence_ids.to(device),
            step_ids=objective_data["step_ids"],
            predictions=q_head(out["z"]),
            backup_value_predictions=backup_value,
            backup_action_predictions=backup_value,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return (loss, metrics)

## Run

Each cycle collects `EPISODES_PER_STREAM` complete tasks per stream, then runs
`TRAIN_STEPS` updates once `LEARNING_STARTS` environment outputs have been collected.
Replay accumulates across cycles. Evaluate the checkpoint in `15_inference.ipynb`.


In [ ]:
env_steps = 0
grad_steps = 0
for cycle in range(NUM_CYCLES):
    env_steps = run_rollout(
        env_steps=env_steps, num_episodes=EPISODES_PER_STREAM, max_cache=MAX_CONTEXT
    )
    print(f"cycle={cycle} rollout  env_step={env_steps}")
    if env_steps >= LEARNING_STARTS:
        loss, metrics = run_train(num_steps=TRAIN_STEPS)
        grad_steps += TRAIN_STEPS
        print(
            f"cycle={cycle} train    loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}  grad_step={grad_steps}",
        )
        if device.type == "cuda":
            torch.cuda.empty_cache()

loader.close()
env.close()
print(
    f"Online training finished ({grad_steps} optimizer steps, {env_steps} env steps)."
)

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")